In [1]:
# 1️⃣ Install dependencies (run once in Jupyter)
!pip install torch torchvision torchaudio
!pip install sentence-transformers

In [2]:
# 2️⃣ Imports
import json
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

/opt/conda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
# 3️⃣ Device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [10]:
# 4️⃣ Load KB dataset
path = "/home/jovyan/work/Cardiology_Data/miriad_cardiology.json"

with open(path, "r") as f:
    raw_data = json.load(f)

# Take first 276139 questions for training
kb_questions = [row["question"] for row in raw_data[:50000] if "question" in row]
print(f"Loaded {len(kb_questions)} questions for training")

Loaded 50000 questions for training


In [13]:
# 5️⃣ Projection Layer
class ProjectionLayer(nn.Module):
    def __init__(self, in_dim=1024, out_dim=768):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)

projector = ProjectionLayer().to(device)

In [14]:
# 6️⃣ Dataset Class
class ProjectionDataset(Dataset):
    def __init__(self, questions, inbedder_model, instructor_model):
        self.questions = questions
        self.inbedder = inbedder_model
        self.instructor = instructor_model

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        text = self.questions[idx]

        # Embed without autograd
        with torch.no_grad():
            inb_emb = self.inbedder.encode(
                text,
                convert_to_tensor=True,
                normalize_embeddings=True
            ).detach()

            inst_emb = self.instructor.encode(
                f"Represent the cardiology question:\n{text}",
                convert_to_tensor=True,
                normalize_embeddings=True
            ).detach()

        return inb_emb.float(), inst_emb.float()

In [15]:
# 7️⃣ Load embedding models
INSTRUCTOR_MODEL_PATH = "hkunlp/instructor-large"
INBEDDER_PATH = "models/inbedder-roberta-large"

inbedder_model = SentenceTransformer(INBEDDER_PATH, device=device)
instructor_model = SentenceTransformer(INSTRUCTOR_MODEL_PATH, device=device)

In [16]:
# 8️⃣ Prepare dataset + dataloader
dataset = ProjectionDataset(kb_questions, inbedder_model, instructor_model)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [17]:
# 9️⃣ Training loop
optimizer = torch.optim.Adam(projector.parameters(), lr=1e-4)
loss_fn = nn.CosineEmbeddingLoss()

for epoch in range(5):
    epoch_loss = 0

    for inb_emb, inst_emb in loader:
        inb_emb = inb_emb.to(device)
        inst_emb = inst_emb.to(device)

        projected = projector(inb_emb)

        target = torch.ones(inb_emb.size(0)).to(device)

        loss = loss_fn(projected, inst_emb, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Avg Loss: {epoch_loss/len(loader):.4f}")

Epoch 1, Avg Loss: 0.0555
Epoch 2, Avg Loss: 0.0330
Epoch 3, Avg Loss: 0.0302
Epoch 4, Avg Loss: 0.0287
Epoch 5, Avg Loss: 0.0278


In [19]:
# 10️⃣ Save trained projector
SAVE_PATH = "/home/jovyan/work/models/projector-new.pt"

torch.save(projector.state_dict(), SAVE_PATH)

print("✅ Projection layer trained and saved at:", SAVE_PATH)

✅ Projection layer trained and saved at: /home/jovyan/work/models/projector-new.pt
